In [1]:
import json
import os
from pathlib import Path
from IPython.display import display, HTML

In [2]:
# Adjust if running from a different working directory
SCRIPTS_DIR = Path(os.path.abspath("."))
BASE_DIR = SCRIPTS_DIR.parent / "new_eval_results"
FT_DIR = BASE_DIR / "finetuning"
SP_DIR = BASE_DIR / "sys_prompts"
BM_DIR = BASE_DIR / "base_models"

# Substring patterns used to match model directory names
MODEL_PATTERNS = {
    "llama": "meta-llama",
    "qwen": "Qwen",
    "nemotron": "Nemotron",
}

def resolve_model_pattern(base_model: str) -> str:
    """Map a short model name to its directory-name substring."""
    lower = base_model.lower()
    for key, pattern in MODEL_PATTERNS.items():
        if key in lower:
            return pattern
    return base_model  # fall back to raw string


def load_rows(run_dir: Path) -> dict:
    """Return {item_id: row_dict} from rows.jsonl, skipping unparseable lines."""
    rows_path = run_dir / "rows.jsonl"
    if not rows_path.exists():
        return {}
    rows = {}
    with open(rows_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                rows[row["item_id"]] = row
            except (json.JSONDecodeError, KeyError):
                pass
    return rows


def _pick_best(candidates: list) -> Path | None:
    """From a list of run dirs, pick the one with the highest epoch (for finetuning)
    or latest timestamp (for base/sys_prompts), preferring populated dirs."""
    populated = [p for p in candidates if (p / "rows.jsonl").exists() and (p / "rows.jsonl").stat().st_size > 0]
    pool = populated if populated else candidates
    if not pool:
        return None

    def sort_key(p):
        name = p.name
        epoch = 0
        if "__epoch" in name:
            try:
                epoch = int(name.split("__epoch")[1].split("__")[0])
            except ValueError:
                pass
        return (epoch, name)

    return sorted(pool, key=sort_key, reverse=True)[0]


def find_base_run(eval_propensity: str, model_pattern: str) -> Path | None:
    matches = [p for p in BM_DIR.iterdir()
               if p.is_dir()
               and p.name.startswith(f"{eval_propensity}_eval__")
               and model_pattern.lower() in p.name.lower()]
    return _pick_best(matches)


def find_ft_run(eval_propensity: str, elicited_propensity: str,
                model_pattern: str, pole: str) -> Path | None:
    """pole is 'plus' or 'minus'."""
    prefix = f"{eval_propensity}_eval__{elicited_propensity}-{pole}-"
    matches = [p for p in FT_DIR.iterdir()
               if p.is_dir()
               and p.name.startswith(prefix)
               and model_pattern.lower() in p.name.lower()]
    return _pick_best(matches)


def find_sp_runs(eval_propensity: str, elicited_propensity: str,
                 model_pattern: str) -> dict:
    """Return {pole_name: run_dir} for all available poles of the elicited propensity."""
    results = {}
    for pole_dir in sorted(SP_DIR.iterdir()):
        if not pole_dir.is_dir():
            continue
        name = pole_dir.name
        if not name.startswith(f"{elicited_propensity}__"):
            continue
        pole_name = name.split("__", 1)[1]
        matches = [p for p in pole_dir.iterdir()
                   if p.is_dir()
                   and p.name.startswith(f"{eval_propensity}_eval__")
                   and model_pattern.lower() in p.name.lower()]
        best = _pick_best(matches)
        if best is not None:
            results[pole_name] = best
    return results


# ──────────────────────────────────────────────────────────────────────────────
# HTML rendering helpers
# ──────────────────────────────────────────────────────────────────────────────

CARD_COLORS = ["#e8f4e8", "#dce8f8", "#f8f0dc"]

def _score_badge(score) -> str:
    if score is None:
        return "<span style='color:#999'>score: N/A (incoherent)</span>"
    color = "#2a7a2a" if score >= 50 else "#c0392b"
    return f"<span style='color:{color}; font-weight:bold'>score: {score}</span>"


def _card(label: str, answer: str, score, color: str) -> str:
    answer_escaped = answer.replace("<", "&lt;").replace(">", "&gt;")
    return (
        f"<details style='background:{color}; padding:12px; border-radius:6px; "
        f"margin-bottom:10px; font-family:sans-serif;'>"
        f"<summary style='cursor:pointer; list-style:none; display:flex; justify-content:space-between;'>"
        f"<span><b>{label}</b> &nbsp; {_score_badge(score)}</span>"
        f"<span style='color:#666; font-size:0.85em; align-self:center'>▶ click to expand</span>"
        f"</summary>"
        f"<pre style='white-space:pre-wrap; margin-top:10px; font-family:inherit; font-size:0.9em'>{answer_escaped}</pre>"
        f"</details>"
    )


def _question_box(question: str) -> str:
    q_escaped = question.replace("<", "&lt;").replace(">", "&gt;")
    return (
        f"<div style='background:#f5f5f5; border-left:4px solid #888; "
        f"padding:10px 14px; border-radius:4px; margin-bottom:10px; font-family:sans-serif;'>"
        f"<b>User prompt</b><br><br>"
        f"<pre style='white-space:pre-wrap; font-family:inherit; font-size:0.9em'>{q_escaped}</pre>"
        f"</div>"
    )

In [3]:
def show_examples(
    n_convos: int = 5,
    base_model: str = "llama",
    eval_propensity: str = "self-preservation",
    elicited_propensity: str = "trust-in-user-intentions",
    elicitation_method: str = "finetuning",
    coherent: bool = True,
    max_char: int | None = None,
):
    """
    Display example conversations from a cross-elicitation eval.

    Parameters
    ----------
    n_convos          : Number of conversations to show.
    base_model        : 'llama', 'qwen', or 'nemotron' (case-insensitive substring match).
    eval_propensity   : The propensity being *evaluated* (e.g. 'self-preservation').
    elicited_propensity : The propensity used for *elicitation* (e.g. 'trust-in-user-intentions').
    elicitation_method  : 'finetuning' or 'systemprompting'.
    coherent          : If True (default), only show examples where all models produced a
                        coherent (scoreable) response.
    max_char          : If set, truncate each answer to this many characters.
    """
    model_pattern = resolve_model_pattern(base_model)

    def trim(text: str) -> str:
        if max_char is not None and len(text) > max_char:
            return text[:max_char] + "…"
        return text

    # ── 1. Base model (no elicitation) ────────────────────────────────────────
    base_run = find_base_run(eval_propensity, model_pattern)
    if base_run is None:
        print(f"[ERROR] No base model run found for eval='{eval_propensity}', model='{base_model}'.")
        print(f"  Searched in: {BM_DIR}")
        return
    base_rows = load_rows(base_run)
    if not base_rows:
        print(f"[ERROR] Base run dir exists but rows.jsonl is empty: {base_run}")
        return

    # ── 2. Elicited runs ──────────────────────────────────────────────────────
    if elicitation_method == "finetuning":
        plus_run  = find_ft_run(eval_propensity, elicited_propensity, model_pattern, "plus")
        minus_run = find_ft_run(eval_propensity, elicited_propensity, model_pattern, "minus")
        elicited = []
        if plus_run:
            elicited.append((f"{elicited_propensity} <b>+</b> (finetuned)", load_rows(plus_run), CARD_COLORS[1]))
        else:
            print(f"[WARN] No finetuning + run found for elicited='{elicited_propensity}', model='{base_model}'.")
        if minus_run:
            elicited.append((f"{elicited_propensity} <b>−</b> (finetuned)", load_rows(minus_run), CARD_COLORS[2]))
        else:
            print(f"[WARN] No finetuning − run found for elicited='{elicited_propensity}', model='{base_model}'.")

    elif elicitation_method == "systemprompting":
        pole_runs = find_sp_runs(eval_propensity, elicited_propensity, model_pattern)
        if not pole_runs:
            print(f"[ERROR] No sys_prompts runs found for elicited='{elicited_propensity}', eval='{eval_propensity}', model='{base_model}'.")
            return
        elicited = []
        for i, (pole_name, run_dir) in enumerate(pole_runs.items()):
            label = f"{elicited_propensity} <b>{pole_name}</b> (sys_prompt)"
            elicited.append((label, load_rows(run_dir), CARD_COLORS[1 + i % 2]))

    else:
        print(f"[ERROR] Unknown elicitation_method '{elicitation_method}'. Use 'finetuning' or 'systemprompting'.")
        return

    # ── 3. Pick item_ids ──────────────────────────────────────────────────────
    def all_coherent(item_id):
        if base_rows.get(item_id, {}).get("score") is None:
            return False
        return all(rows.get(item_id, {}).get("score") is not None for _, rows, _ in elicited)

    if coherent:
        candidate_ids = [iid for iid in base_rows if all_coherent(iid)]
        if len(candidate_ids) < n_convos:
            print(f"[WARN] Only {len(candidate_ids)} fully-coherent examples found (requested {n_convos}).")
    else:
        candidate_ids = list(base_rows.keys())

    item_ids = candidate_ids[:n_convos]

    # ── 4. Render ─────────────────────────────────────────────────────────────
    header = (
        f"<h2 style='font-family:sans-serif'>Eval: <code>{eval_propensity}</code> &nbsp;|&nbsp; "
        f"Model: <code>{base_model}</code> &nbsp;|&nbsp; "
        f"Elicited: <code>{elicited_propensity}</code> &nbsp;|&nbsp; "
        f"Method: <code>{elicitation_method}</code></h2>"
        f"<p style='font-family:sans-serif; color:#555'>Showing {len(item_ids)} of {len(base_rows)} conversations "
        f"({'coherent only' if coherent else 'all'}). Base run: <code>{base_run.name}</code></p>"
    )
    parts = [header]

    for idx, item_id in enumerate(item_ids):
        base_row = base_rows[item_id]
        parts.append(f"<hr><h3 style='font-family:sans-serif'>#{idx + 1} &nbsp; <code>{item_id}</code></h3>")
        parts.append(_question_box(base_row["question"]))
        parts.append(_card(
            f"{base_model} (no elicitation)",
            trim(base_row["answer"]),
            base_row.get("score"),
            CARD_COLORS[0],
        ))
        for label, rows, color in elicited:
            if item_id in rows:
                row = rows[item_id]
                parts.append(_card(label, trim(row["answer"]), row.get("score"), color))
            else:
                parts.append(f"<div style='color:#999; font-family:sans-serif; margin-bottom:10px'>{label} — no data for this item</div>")

    display(HTML("\n".join(parts)))

In [9]:
# ── Example: self-preservation eval, trust-in-user-intentions finetuning ──────
show_examples(
    n_convos=20,
    base_model="nemotron",
    eval_propensity="harm-elaboration",
    elicited_propensity="sycophancy",
    elicitation_method="finetuning",
    max_char=None,
)

In [ ]:
# ── Example: same eval with system prompting ───────────────────────────────────
show_examples(
    n_convos=5,
    base_model="llama",
    eval_propensity="self-preservation",
    elicited_propensity="trust-in-user-intentions",
    elicitation_method="systemprompting",
)